# Embeddings y LSTM mínimo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohtar10/icesi-nlp/blob/main/Sesion1/3-embeddings-y-lstm-minimo.ipynb)

Este notebook conecta dos ideas clave de la transición hacia deep learning para NLP: primero observamos cómo los embeddings capturan similitud semántica y luego construimos un clasificador LSTM pequeño, explícito y fácil de seguir. El objetivo es preparar el terreno para transformers sin introducir desde el inicio demasiada infraestructura experimental.

## Referencias
* [Word Vectors and spaCy Similarity](https://spacy.io/usage/linguistic-features#vectors-similarity)
* [Long Short-Term Memory](https://www.researchgate.net/publication/13853244_Long_Short-Term_Memory#fullTextFileContent)
* Dataset: https://huggingface.co/datasets/mteb/spanish_news

## Preparación del entorno
Asumiendo que la librería ya se encuentra instalada, dependiendo de la tarea, necesitamos descargar el modelo o las dependencias puntuales antes de empezar.


In [1]:
import warnings

warnings.filterwarnings('ignore')

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [2]:
!test '{IN_COLAB}' = 'True' && wget  https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/requirements.txt && pip install -r requirements.txt

In [3]:
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 35.2 MB/s  0:00:08:00:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')


In [4]:
import spacy

nlp = spacy.load('en_core_web_lg')

En esta ocasión, este modelo de spaCy más grande ya contiene vectores de palabras densos de 300 dimensiones que podemos utilizar de inmediato.

Por ejemplo, exploremos la palabra `lion`

In [5]:
nlp(u'lion').vector

array([ 1.8963e-01, -4.0309e-01,  3.5350e-01, -4.7907e-01, -4.3311e-01,
        2.3857e-01,  2.6962e-01,  6.4332e-02,  3.0767e-01,  1.3712e+00,
       -3.7582e-01, -2.2713e-01, -3.5657e-01, -2.5355e-01,  1.7543e-02,
        3.3962e-01,  7.4723e-02,  5.1226e-01, -3.9759e-01,  5.1333e-03,
       -3.0929e-01,  4.8911e-02, -1.8610e-01, -4.1702e-01, -8.1639e-01,
       -1.6908e-01, -2.6246e-01, -1.5983e-02,  1.2479e-01, -3.7276e-02,
       -5.7125e-01, -1.6296e-01,  1.2376e-01, -5.5464e-02,  1.3244e-01,
        2.7519e-02,  1.2592e-01, -3.2722e-01, -4.9165e-01, -3.5559e-01,
       -3.0630e-01,  6.1185e-02, -1.6932e-01, -6.2405e-02,  6.5763e-01,
       -2.7925e-01, -3.0450e-03, -2.2400e-02, -2.8015e-01, -2.1975e-01,
       -4.3188e-01,  3.9864e-02, -2.2102e-01, -4.2693e-02,  5.2748e-02,
        2.8726e-01,  1.2315e-01, -2.8662e-02,  7.8294e-02,  4.6754e-01,
       -2.4589e-01, -1.1064e-01,  7.2250e-02, -9.4980e-02, -2.7548e-01,
       -5.4097e-01,  1.2823e-01, -8.2408e-02,  3.1035e-01, -6.33

In [6]:
nlp(u'lion').vector.shape

(300,)

Como podemos observar, `lion` es representada por un vector de *300* dimensiones. Esto es lo que llamamos un `Word to Vector` o *Word2Vec*.

Sin embargo, esta no es la única característica de los modelos de spaCy. También podemos tener un vector de documento lo que significa que por cada documento vamos a promediar los vectores de cada vector que lo compone.

In [7]:
nlp(u'The quick brown fox jumped').vector.shape

(300,)

## Comprobando la similitud entre tokens

Recordemos que una de las características más importantes de estos vectores es que desbloquean exitosamente la semantica de las palabras con operaciones aritméticas, tarea que antes era muy difícil con las técnicas más clásicas.

Gracias a que ya contamos con vectores pre-computados, podemos medir la similitud de estas palabras. La similitud se mide con el coseno entre los vectores. 

El rango del coseno es $[-1,+1]$, donde valores cercanos a cero, indican que los vectores son ortogonales, es decir su ángulo es de 90 grados y se consideran disimilares. Cuando la similitud es positiva cercano a $1$, quiere decir que ambos vecrtores son similares, ya que su angulo es inferior a 90 grados. Finalmente, cuando la similitud es negativa, cercana a $-1$, quiere decir que los vectores son opuestos, su ángulo es mayor a 90 grados. Esto último es diferente a valores ceranos a 0 ya que lo opuesto también lleva consigo una relación semántica.

In [8]:
tokens = nlp('lion cat pet')

Por ejemplo, podemos establecer una relación semántica entre un león y un gato, amobos son felinos. Además, entre un gato y una mascota, ya que un gato es usualmente una. Ahora verifiquemos esta suposición aritméticamente:

In [9]:
def print_similarity(tokens):
    for token in tokens:
        sim = {t:token.similarity(t) for t in tokens}
        print(f'Similarities with word {token}:\n{sim}\n')

print_similarity(tokens)

Similarities with word lion:
{lion: 1.0, cat: 0.5265437960624695, pet: 0.39923766255378723}

Similarities with word cat:
{lion: 0.5265437960624695, cat: 1.0, pet: 0.7505456805229187}

Similarities with word pet:
{lion: 0.39923766255378723, cat: 0.7505456805229187, pet: 1.0}



Naturalmente, cada palabra es completamente similar con sigo misma, por eso para lion, vemos que la similitud es 1. Nótese que la palabra león tiene una similitud de 0.5 con la palabra gato, pero al mismo tiempo tiene una baja similitud con la palabra mascota. 

Entonces verdaderamente la similitud entre los vectores puede revelar una relación semántica entre ellos. Y lo mejor es que es sistemáticamente computable con aritmética vectorial.

Observemos otros ejemplos:

In [10]:
tokens = nlp('like love hate')

Desde nuestro conocimeinto, sabemos que `love` (amor) y `hate` (odio) son antonimos, es decir, palabras con significado opuesto. Pero como ambas tienden a ser utilizadas en el mismo contexto, sus vectores pueden tener algo de similitud. Esto significa que los vectores son buenos para detectar contexto pero no para obtener definiciones u obtener un significado humanamente entendible de las palabras.

In [11]:
print_similarity(tokens)

Similarities with word like:
{like: 1.0, love: 0.6579040288925171, hate: 0.6574651598930359}

Similarities with word love:
{like: 0.6579040288925171, love: 1.0, hate: 0.6393099427223206}

Similarities with word hate:
{like: 0.6574651598930359, love: 0.6393099427223206, hate: 1.0}



Ahora, podemos inspeccionar el vocabulario de spaCy:

In [12]:
nlp.vocab.vectors.shape

(342918, 300)

Tenemos un total de $342,918$ palabras con $300$ dimensiones cada una.

Si una palabra no está presente en el vocabulario, significa que no va a tener un vector.

In [13]:
tokens = nlp('dog cat nargle')
for token in tokens:
    print(token.text, token.has_vector, token.vector_norm, token.is_oov)

dog True 7.0336733 False
cat True 6.6808186 False
nargle False 0.0 True


Esto debemos tenerlo en cuenta sobretodo a la hora de usar modelos pre-entrenados, si sabemos de antemano que nuestro corpus tiene tokens que no están en el vocabulario, pues lo más seguro es que nuestro modelo no sea lo suficientemente bueno.

## Calculando un nuevo vector
Técnicamente, podemos usar algebra lineal para obtener un vector de una nueva palabra. Veamos el ejemplo clásico de los word embeddings, la relación entre rey, hombre, mujer y reina:

In [14]:
from scipy import spatial

cosine_similarity = lambda v1, v2: 1 - spatial.distance.cosine(v1, v2)

In [15]:
king = nlp.vocab['king'].vector
man = nlp.vocab['man'].vector
woman = nlp.vocab['woman'].vector

Ahora podemos realizar operaciones simples como adición y substracción de vectores y ver lo que nos deja.

Si a `king` le restamos `man` y le sumamos `woman`, quizás nuestra intuición nos diga que el resultado debería ser la palabra `reina` verdad?

In [16]:
# Notice for king we substract the man features and add the women features.
# We expect the resulting vector to be similar to Queen, princess, highness, etc.
nv = king - man + woman

In [17]:
cosine_similarity(nv, nlp.vocab['queen'].vector)

np.float32(0.78808445)

In [18]:
computed_similarities = {word.text:cosine_similarity(nv, word.vector) 
                         for word in nlp.vocab if word.has_vector and word.is_lower and word.is_alpha
                        }


In [19]:
computed_similarities = sorted(computed_similarities.items(), key=lambda kv: kv[1], reverse=True)

In [20]:
computed_similarities[:10]

[('king', np.float32(0.802426)),
 ('queen', np.float32(0.78808445)),
 ('woman', np.float32(0.5150813)),
 ('she', np.float32(0.39561844)),
 ('lion', np.float32(0.38601506)),
 ('who', np.float32(0.31594998)),
 ('fox', np.float32(0.30404305)),
 ('brown', np.float32(0.28812397)),
 ('when', np.float32(0.28596795)),
 ('dare', np.float32(0.28260583))]

Observemos la medida de similitud para la palabra `queen`, que es significativamente cercana a lo que queríamos.

## De los embeddings a las secuencias

Los embeddings de spaCy son útiles para entender similitud semántica, pero cuando pasamos a clasificación de texto necesitamos un modelo que procese secuencias completas. En esta sección construiremos una versión mínima de un clasificador con LSTM sobre un subconjunto del dataset de noticias en español. La idea no es maximizar la métrica, sino entender el flujo completo con la menor cantidad de abstracciones posible.

In [22]:
from datasets import load_dataset
import torch
import warnings
import os

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
subset_size = 6000 if torch.cuda.is_available() else 3000
dataset = load_dataset('mteb/spanish_news', split=f'train[:{subset_size}]')
print(dataset)
print(f'Dispositivo: {device}')
print(f'Registros cargados: {len(dataset)}')


Generating test split: 100%|██████████| 2048/2048 [00:00<00:00, 76829.61 examples/s]

Dataset({
    features: ['language', 'label', 'newspaper', 'hash', 'text'],
    num_rows: 6000
})
Dispositivo: cuda
Registros cargados: 6000


Trabajaremos con un subconjunto para que el notebook sea razonable de ejecutar en CPU o en una GPU pequeña. Más adelante, en el notebook avanzado, recuperaremos un pipeline más completo con entrenamiento más largo, logging y utilidades adicionales.

In [23]:
from collections import Counter
import re

def simple_tokenizer(text: str):
    text = text.lower()
    text = re.sub(r'[^a-záéíóúüñ0-9]+', ' ', text)
    return text.strip().split()

token_counts = Counter()
for text in dataset['text']:
    token_counts.update(simple_tokenizer(text))

max_vocab = 20000
top_tokens = [token for token, _ in token_counts.most_common(max_vocab - 2)]
vocab = {'[PAD]': 0, '[UNK]': 1}
for token in top_tokens:
    vocab[token] = len(vocab)

def tokenize_text(text: str, max_length: int = 256):
    token_ids = [vocab.get(token, vocab['[UNK]']) for token in simple_tokenizer(text)[:max_length]]
    token_ids += [vocab['[PAD]']] * (max_length - len(token_ids))
    return token_ids

print(f'Vocabulario construido: {len(vocab)} tokens')
print(tokenize_text('hola mundo desde el curso', max_length=12))


Vocabulario construido: 20000 tokens
[11880, 99, 41, 5, 1162, 0, 0, 0, 0, 0, 0, 0]


La idea del padding es que todas las secuencias lleguen al modelo con la misma longitud. Esto simplifica el entrenamiento por lotes y deja visible qué parte del pipeline es puramente preparación de datos.

In [25]:
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split

class SpanishNewsDataset(Dataset):
    def __init__(self, dataset, tokenizer, seq_length: int = 256):
        self.dataset = dataset
        self.tokenizer = tokenizer
        self.seq_length = seq_length
        labels = np.unique(dataset['label'])
        self.id_to_class = dict(enumerate(labels))
        self.class_to_id = {label: idx for idx, label in self.id_to_class.items()}
        self.num_classes = len(self.id_to_class)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset[idx]
        return {
            'input_ids': torch.tensor(self.tokenizer(row['text'], max_length=self.seq_length), dtype=torch.long),
            'label': torch.tensor(self.class_to_id[row['label']], dtype=torch.long),
            'text': row['text'],
        }

seq_length = 256
news_dataset = SpanishNewsDataset(dataset, tokenize_text, seq_length=seq_length)
train_size = int(len(news_dataset) * 0.8)
val_size = int(len(news_dataset) * 0.1)
test_size = len(news_dataset) - train_size - val_size
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset, test_dataset = random_split(news_dataset, [train_size, val_size, test_size], generator=generator)

batch_size = 64 if torch.cuda.is_available() else 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')
print(f'Número de clases: {news_dataset.num_classes}')


Train: 4800 | Val: 600 | Test: 600
Número de clases: 12


## Definición del modelo LSTM

La arquitectura mínima tendrá tres partes: una capa de embeddings entrenable, un bloque LSTM y una capa lineal para clasificar la última representación oculta. Más adelante, el notebook avanzado mostrará cómo encapsular este mismo flujo con herramientas más cómodas para experimentación.

In [26]:
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size: int, num_classes: int, embedding_dim: int = 128, hidden_dim: int = 128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        _, (hidden_state, _) = self.lstm(embedded)
        logits = self.classifier(hidden_state[-1])
        return logits

model = LSTMClassifier(len(vocab), news_dataset.num_classes).to(device)
model


LSTMClassifier(
  (embedding): Embedding(20000, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True)
  (classifier): Linear(in_features=128, out_features=12, bias=True)
)

## Entrenamiento mínimo

Aquí sí escribimos el ciclo de entrenamiento de forma explícita para que el papel de cada parte quede claro. No buscamos una receta de producción, sino una versión suficientemente pequeña para inspeccionarla completa en clase.

In [27]:
from tqdm.auto import tqdm

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 2 if torch.cuda.is_available() else 1

def run_epoch(dataloader, training: bool = True):
    model.train(training)
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for batch in tqdm(dataloader, disable=not training):
        input_ids = batch['input_ids'].to(device)
        labels = batch['label'].to(device)
        logits = model(input_ids)
        loss = criterion(logits, labels)

        if training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += labels.size(0)

    return total_loss / total_examples, total_correct / total_examples

history = []
for epoch in range(epochs):
    train_loss, train_acc = run_epoch(train_loader, training=True)
    with torch.no_grad():
        val_loss, val_acc = run_epoch(val_loader, training=False)
    history.append({'epoch': epoch + 1, 'train_loss': train_loss, 'train_acc': train_acc, 'val_loss': val_loss, 'val_acc': val_acc})
    print(f"Epoch {epoch + 1}: train_loss={train_loss:.4f} train_acc={train_acc:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")


100%|██████████| 75/75 [00:02<00:00, 30.10it/s]


Epoch 1: train_loss=2.4769 train_acc=0.1023 val_loss=2.4704 val_acc=0.0933


100%|██████████| 75/75 [00:02<00:00, 32.28it/s]


Epoch 2: train_loss=2.3509 train_acc=0.2119 val_loss=2.4785 val_acc=0.1300


In [28]:
import pandas as pd

history_df = pd.DataFrame(history)
history_df


,epoch,train_loss,train_acc,val_loss,val_acc
0,1,2.476902,0.102292,2.470383,0.093333
1,2,2.350885,0.211875,2.478496,0.130000


## Evaluación rápida

Una vez entrenado el modelo, medimos correctitud en test y observamos algunas predicciones. Esto nos basta para entender la lógica del baseline antes de dar el salto a transformers.

In [29]:
with torch.no_grad():
    test_loss, test_acc = run_epoch(test_loader, training=False)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_acc:.4f}')


Test loss: 2.5563
Test accuracy: 0.1150


In [31]:
sample_indices = test_dataset.indices[:5]
id_to_token = {idx: token for token, idx in vocab.items()}
rows = []
for sample_idx in sample_indices:
    row = dataset[sample_idx]
    input_ids = torch.tensor([tokenize_text(row['text'], max_length=seq_length)], dtype=torch.long).to(device)
    with torch.no_grad():
        pred_id = model(input_ids).argmax(dim=1).item()
    rows.append({
        'categoría_real': row['label'],
        'categoría_predicha': news_dataset.id_to_class[pred_id],
        'texto': row['text'][:280] + '...'
    })

pd.DataFrame(rows)


,categoría_real,categoría_predicha,texto
0,6,10,Negar que el mundo de las frituras vive en una...
1,9,3,Analizando anillos de crecimiento anual de ant...
2,2,3,"Por suerte para la Unión Deportiva Las Palmas,..."
3,8,3,El mes de febrero nos ha dejado temperaturas m...
4,0,3,"El phishing, un tipo de ciberataque que consis..."


## Conclusiones

- Un embedding entrenable permite convertir tokens a vectores densos antes de alimentar la secuencia a la LSTM.
- Incluso una implementación mínima ya muestra el flujo completo: preparar datos, vectorizar, procesar secuencias y clasificar.
- Si quieres una versión más completa con logging, early stopping y organización experimental, continúa con el notebook `4-lstm-avanzado-con-pytorch-lightning.ipynb` de esta misma sesión.
